## **1. Setup**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_PROC    = PROJECT_ROOT / "data" / "processed"
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"Raw CSVs: {len(list(DATA_RAW.glob('*.csv')))}")

Raw CSVs: 14


In [2]:
from src.etl import load_raw_csvs

df_raw = load_raw_csvs(DATA_RAW)
print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (4076, 4)


,year,month,capacity_hm3,reservoir
0,NaN,enero,"23,891",RioCofio_LaAcena
1,NaN,febrero,"23,816",RioCofio_LaAcena
2,NaN,marzo,"23,245",RioCofio_LaAcena
3,NaN,abril,"23,35",RioCofio_LaAcena
4,NaN,mayo,"23,678",RioCofio_LaAcena


In [3]:
from src.etl import clean_dataframe

df_clean = clean_dataframe(df_raw)
print(f"Shape: {df_clean.shape}")
print(f"Nulls: {df_clean['capacity_hm3'].isna().sum()}")
print(f"Reservoirs: {sorted(df_clean['reservoir'].unique())}")
df_clean.head()

Shape: (3303, 4)
Nulls: 90
Reservoirs: ['RioCofio_LaAcena', 'RioGuadalix_Pedrezuela', 'RioGuadarrama-Aulencia_LaJorosa', 'RioGuadarrama-Aulencia_Navalmedio', 'RioGuadarrama-Aulencia_Valmayor', 'RioJarama_ElVado', 'RioLozoya_ElAtazar', 'RioLozoya_ElVillar', 'RioLozoya_LaPinilla', 'RioLozoya_PuentesViejas', 'RioLozoya_Riosequillo', 'RioManzanares_Navacerrada', 'RioManzanares_Santillana']


,year,month,capacity_hm3,reservoir
0,1998,enero,23.891,RioCofio_LaAcena
1,1998,febrero,23.816,RioCofio_LaAcena
2,1998,marzo,23.245,RioCofio_LaAcena
3,1998,abril,23.350,RioCofio_LaAcena
4,1998,mayo,23.678,RioCofio_LaAcena


In [5]:
from src.etl import build_pivot

df_pivot = build_pivot(df_clean)
print(f"Shape: {df_pivot.shape}")
print(f"Date range: {df_pivot['ds'].min()} → {df_pivot['ds'].max()}")
df_pivot.head()

Shape: (279, 17)
Date range: 1998-01-01 00:00:00 → 2021-03-01 00:00:00


,ds,year,month,total_hm3,RioCofio_LaAcena,RioGuadalix_Pedrezuela,RioGuadarrama-Aulencia_LaJorosa,RioGuadarrama-Aulencia_Navalmedio,RioGuadarrama-Aulencia_Valmayor,RioJarama_ElVado,RioLozoya_ElAtazar,RioLozoya_ElVillar,RioLozoya_LaPinilla,RioLozoya_PuentesViejas,RioLozoya_Riosequillo,RioManzanares_Navacerrada,RioManzanares_Santillana
0,1998-01-01,1998,enero,579.861417,14.735250,25.041000,4.664625,0.383250,86.818250,31.715917,289.552750,18.693167,27.822125,34.653083,45.782,NaN,NaN
1,1998-02-01,1998,febrero,601.215333,16.128000,26.113625,4.938167,0.410417,90.964875,35.158042,295.381500,19.260208,28.420292,37.800208,46.640,NaN,NaN
2,1998-03-01,1998,marzo,640.598417,18.099792,28.409583,5.466333,0.402833,96.294750,37.770958,318.677333,18.960875,30.704625,39.846333,45.965,NaN,NaN
3,1998-04-01,1998,abril,675.753826,19.200478,28.508087,5.687217,0.432652,100.016000,40.119217,342.452652,18.482522,32.192348,42.453652,46.209,NaN,NaN
4,1998-05-01,1998,mayo,685.307652,19.457565,27.067609,5.460783,0.399565,100.451174,37.936261,355.888870,18.611739,32.372522,43.707565,43.954,NaN,NaN


In [6]:
assert df_pivot["total_hm3"].isna().sum() == 0
assert df_pivot["ds"].is_monotonic_increasing

df_pivot.to_csv(DATA_PROC / "reservoirs_pivot.csv", index=False)
df_clean.to_csv(DATA_PROC / "reservoirs_long.csv", index=False)

print(f"Exported reservoirs_pivot.csv — {df_pivot.shape}")
print(f"Exported reservoirs_long.csv  — {df_clean.shape}")

Exported reservoirs_pivot.csv — (279, 17)
Exported reservoirs_long.csv  — (3303, 4)
